<div align="center">

# Week 03: Discrete and Hybrid Dynamics: State Machines, Thermostats and Protection Relays

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-03) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/Week03_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the constants of the library, the battery and the relays, and the spring weekday of Week 1. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

# The campus microgrid: the library, the protection of its feeders and the island
C_KWH_K, R_K_KW, P_HEAT_KW = 30.0, 0.4, 60.0     # library: heat capacity, thermal resistance, heater power
E_KWH, ETA, P_INV_KW = 500.0, 0.95, 250.0        # battery: energy, efficiency of each direction, inverter limit
K_SI, ALPHA_SI = 0.14, 0.02                      # IEC standard inverse curve of the overcurrent relays
# The spring weekday of Week 1: building load and car park in kW, wind speed in m/s, one value per hour
LOAD_KW = np.array([262, 255, 250, 248, 252, 270, 330, 430, 520, 570, 585, 590,
                    575, 580, 585, 575, 560, 530, 520, 500, 450, 390, 320, 280], float)
EV_KW = np.array([0, 0, 0, 0, 0, 0, 0, 22, 66, 66, 44, 22,
                  22, 22, 22, 44, 66, 110, 110, 88, 44, 22, 0, 0], float)
WIND_MS = np.array([7.8, 8.0, 8.2, 8.0, 7.6, 7.0, 6.2, 5.4, 4.8, 4.4, 4.2, 4.6,
                    5.0, 5.2, 5.0, 4.6, 4.2, 3.8, 3.6, 3.5, 3.8, 4.6, 5.8, 7.0])
print("Setup complete. Running in Colab:", IN_COLAB, "| library time constant", C_KWH_K * R_K_KW,
      "h | inverter limit", P_INV_KW, "kW")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 3.1": (lambda: 700, 1e-9),
    "Exercise 3.2": (lambda: 'broken', 0),
    "Exercise 3.3": (lambda: 1269, 1e-9),
    "Exercise A": (lambda: 'B', 0),
    "Exercise B": (lambda: 60 * cycle_times(0.5, 5.0)[0], 1e-3),
    "Exercise C": (lambda: float(t_iec(10, 0.1)), 1e-3),
    "Exercise D": (lambda: kept_kw(21), 1e-3),
    "Exercise E": (lambda: float(STAGE3['SoC at 21:00 that rides through until 23:00']), 1e-6),
}

## Python step 3: States and transitions

A state machine is a table: For each state and each event, it names the next state. This step builds the tools of the week: Dictionaries as tables, tuples as keys, sets of visited states and formatted traces &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#ref-3)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "WIND_MS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-03/"
               "NB03_discrete_hybrid_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Dictionaries as tables

In [ ]:
# Python practice: a small example for learning the language
peak_kw = {"B1": 180, "B2": 120, "B3": 150, "B4": 90, "B5": 160}
print("library:", peak_kw["B2"], "kW")
for name, kw in peak_kw.items():
    print(f"{name}: {kw:>4} kW")
print("unknown building:", peak_kw.get("B9", 0), "kW")

A dictionary maps keys to values. The expression `peak_kw["B2"]` reads one value, and the method `.items()` gives the pairs of key and value for a loop. The method `.get(key, default)` returns the default when the key is missing, instead of an error.

### Tuples as keys: A transition table

In [ ]:
# Python practice: a small example for learning the language
LAMP = {("off", "press"): "on", ("on", "press"): "off", ("on", "fail"): "broken"}
state = "off"
for event in ["press", "press", "press", "fail", "press"]:
    state = LAMP.get((state, event), state)      # no entry: the state stays
    print(f"{event:<6} -> {state}")

A tuple such as `("off", "press")` can serve as a key, which turns a dictionary into a transition table. `LAMP.get((state, event), state)` returns the next state, or the current state when the table has no entry. Events without a transition are thus ignored.

### Sets: The states that can be reached

In [ ]:
# Python practice: a small example for learning the language
reached, frontier = {"off"}, ["off"]
while frontier:
    s = frontier.pop()
    for (src, ev), dst in LAMP.items():
        if src == s and dst not in reached:
            reached.add(dst)
            frontier.append(dst)
print("reachable from off:", sorted(reached))

A set holds each element once, and `in` tests membership quickly. The `while` loop takes a state from the list `frontier` and adds every state that one transition reaches. It stops when no new state appears. The lamp can reach all three states, and `broken` has no way back.

### Formatted traces

In [ ]:
# Python practice: a small example for learning the language
steps = [(0, "off"), (5, "on"), (65, "off"), (130, "on")]
for minute, s in steps:
    print(f"{minute // 60:02d}:{minute % 60:02d}  {s:<4}  {'#' * (minute // 10)}")

In an f-string, `{s:<4}` pads a text to four characters and `{x:02d}` writes an integer with two digits. The operator `//` divides integers without the remainder, and `%` gives the remainder. Together they turn minutes into hours and minutes.

**Quick check.** What does `LAMP.get(("broken", "press"), "broken")` return, and why?

<details><summary>Answer</summary>

`"broken"`: The table has no entry for this pair, so the default, the current state, is returned.

</details>

### Exercises for Python step 3

**Exercise 3.1.** Store the sum of the peak loads in `peak_kw` in `total_peak`, using `.values()`.

In [ ]:
total_peak = None   # your answer

check("Exercise 3.1", total_peak)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
total_peak = sum(peak_kw.values())

check("Exercise 3.1", total_peak)

**Exercise 3.2.** Start the lamp in state `"on"` and apply the events `"fail"` and `"press"` with `LAMP.get`. Store the final state in `after`.

In [ ]:
after = None   # your answer

check("Exercise 3.2", after)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
after = "on"
for ev in ["fail", "press"]:
    after = LAMP.get((after, ev), after)

check("Exercise 3.2", after)

**Exercise 3.3.** Convert the time 21:09 into minutes after midnight and store the number in `n_minutes`.

In [ ]:
n_minutes = None   # your answer

check("Exercise 3.3", n_minutes)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_minutes = 21 * 60 + 9

check("Exercise 3.3", n_minutes)

## 1. Finite state machines: The charger of the car park &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#ref-1)&#93;

*Lecture: [Finite state machines: Modes and events](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#finite-state-machines)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "reached" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-03/"
               "NB03_discrete_hybrid_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A charger of the car park is always in one of a few states, and only events move it from one to the next. Python writes its transition table as a dictionary, runs sequences of events through it and checks a safety property for all states that can be reached.

**From the formulas to the code.** The transition function $\delta(s, e) = s'$ is the dictionary `CHARGER`, with pairs of state and event as keys. The output of each state, the voltage of the control pilot and the contactor, is the dictionary `OUTPUT`. `run(table, s0, events)` returns the trace of states, and `reachable(table, s0)` the set of states that some sequence of events reaches.

In [ ]:
CHARGER = {  # (state, event): next state; the states follow the control pilot of IEC 61851-1
    ("A", "plug"): "B", ("A", "fault"): "F",
    ("B", "ready"): "C", ("B", "unplug"): "A", ("B", "fault"): "F",
    ("C", "full"): "B", ("C", "unplug"): "A", ("C", "fault"): "F",
    ("F", "reset"): "A"}
OUTPUT = {"A": (12, "open"), "B": (9, "open"), "C": (6, "closed"), "F": (-12, "open")}   # pilot volts, contactor
EVENTS = ["plug", "ready", "full", "unplug", "fault", "reset"]

def run(table, s0, events):
    trace = [s0]
    for e in events:
        trace.append(table.get((trace[-1], e), trace[-1]))   # no entry: the event is ignored
    return trace

def reachable(table, s0):
    seen, todo = {s0}, [s0]
    while todo:
        s = todo.pop()
        for (src, e), dst in table.items():
            if src == s and dst not in seen:
                seen.add(dst)
                todo.append(dst)
    return seen

events = ["plug", "ready", "full", "ready", "unplug"]
trace = run(CHARGER, "A", events)
print("trace:", " -> ".join(trace))
closings = sum(1 for a, b in zip(trace, trace[1:]) if OUTPUT[a][1] == "open" and OUTPUT[b][1] == "closed")
print("contactor closings:", closings)
print("table entries:", len(CHARGER), "of", len(OUTPUT) * len(EVENTS), "pairs of state and event")
print("reachable from A:", sorted(reachable(CHARGER, "A")))
safe = all((OUTPUT[s][1] == "closed") == (s == "C") for s in reachable(CHARGER, "A"))
print("the contactor is closed in state C and only there:", safe)
print(f"duty cycle of the pilot for 32 A: {32 / 0.6:.1f} percent")

**Python note.** `zip(trace, trace[1:])` pairs each state with the next one, so the sum counts the steps from an open to a closed contactor. `all(...)` is `True` only when the condition holds for every reachable state. The last line uses the rule of the standard that the current equals the duty cycle times 0.6 A.

**Exercise A.** Run the events `plug`, `fault`, `ready`, `reset` and `plug` from state `"A"` with `run` and store the final state in `final_a`.

In [ ]:
final_a = None   # your answer

check("Exercise A", final_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
final_a = run(CHARGER, "A", ["plug", "fault", "ready", "reset", "plug"])[-1]

check("Exercise A", final_a)

In [ ]:
# Figure: the voltage of the control pilot along the trace of the worked example
volts = [OUTPUT[s][0] for s in trace]
fig, ax = plt.subplots(figsize=(8, 3.0))
ax.step(range(len(volts)), volts, where="post", color="#1E6FA8", lw=2)
for i, s in enumerate(trace):
    ax.annotate(s, (i + 0.4, volts[i] + 0.6), fontsize=10)
ax.set_xticks(range(len(trace)))
ax.set_xticklabels(["start"] + events)
ax.set_ylim(-1, 14)
ax.set_xlabel("event")
ax.set_ylabel("pilot voltage (V)")
plt.show()

**Observe and reflect.** Why does the table ignore the event `full` in state B? What could go wrong if a charger closed its contactor in state B, before the car is ready?

## 2. The thermostat of the library as a hybrid automaton &#91;[5](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#ref-5)&#93;

*Lecture: [Hybrid automata: The thermostat of the library](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#hybrid-automata)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "reachable" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-03/"
               "NB03_discrete_hybrid_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The thermostat of the library switches the heater on and off, and between the switchings the temperature follows the model of Week 2. Python simulates this hybrid automaton, detects its switchings and compares the cycle with the formulas of the lecture.

**From the formulas to the code.** The flows $C\,\dot{T} = (T_{out} - T)/R$ in the mode OFF and $C\,\dot{T} = (T_{out} - T)/R + P$ in the mode ON are the body of `thermostat`. The guards $T \le T_{lo}$ and $T \ge T_{hi}$ switch the mode. `cycle_times(band, T_out)` computes $t_{on} = \tau \ln\frac{T_{ss} - T_{lo}}{T_{ss} - T_{hi}}$ and $t_{off} = \tau \ln\frac{T_{hi} - T_{out}}{T_{lo} - T_{out}}$.

In [ ]:
def thermostat(band, T_out, hours=24, T0=21.0, T_set=21.0, dt_h=1 / 60):
    """Hybrid automaton with the modes OFF (0) and ON (1); returns time, temperature, mode and switching times."""
    lo, hi = T_set - band / 2, T_set + band / 2
    n = int(round(hours / dt_h))
    T, mode, switches = np.empty(n + 1), np.zeros(n + 1, int), []
    T[0], m = T0, 0
    for k in range(n):
        if m == 0 and T[k] <= lo:          # guard OFF -> ON
            m = 1
            switches.append(k * dt_h)
        elif m == 1 and T[k] >= hi:        # guard ON -> OFF
            m = 0
            switches.append(k * dt_h)
        mode[k] = m
        T[k + 1] = T[k] + dt_h * ((T_out - T[k]) / R_K_KW + m * P_HEAT_KW) / C_KWH_K
    mode[n] = m
    return np.arange(n + 1) * dt_h, T, mode, switches

def cycle_times(band, T_out, T_set=21.0):
    tau, lo, hi = C_KWH_K * R_K_KW, T_set - band / 2, T_set + band / 2
    T_ss_on = T_out + R_K_KW * P_HEAT_KW
    t_on = tau * math.log((T_ss_on - lo) / (T_ss_on - hi))
    t_off = tau * math.log((hi - T_out) / (lo - T_out))
    return t_on, t_off

t_on, t_off = cycle_times(1.0, 5.0)
print(f"formula: on {60 * t_on:.1f} min, off {60 * t_off:.1f} min, duty {t_on / (t_on + t_off):.3f}, "
      f"{48 / (t_on + t_off):.1f} switchings a day")
t, T, mode, sw = thermostat(1.0, 5.0, hours=48)
day2 = t >= 24                                         # the second day, after the start
print(f"simulation, second day: {sum(1 for s in sw if s >= 24)} switchings, mean power "
      f"{P_HEAT_KW * mode[day2].mean():.1f} kW, temperature from {T[day2].min():.2f} to {T[day2].max():.2f} C")

**Python note.** The mode is an integer, 0 for OFF and 1 for ON, so `m * P_HEAT_KW` switches the power in the flow. The guards are checked before each Euler step, as a controller reads its sensor once a minute. The temperature therefore passes a threshold by a few hundredths of a degree before the mode changes.

**Exercise B.** Store the on time of the heater, in minutes, for a band of 0.5 K at 5 deg C outside in `t_on_b`.

In [ ]:
t_on_b = None   # your answer

check("Exercise B", t_on_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_on_b = 60 * cycle_times(0.5, 5.0)[0]

check("Exercise B", t_on_b)

In [ ]:
# Figure: the thermostat of the library over one day, and the switchings against the band
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(t[day2] - 24, T[day2], color="#1E6FA8")
ax[0].fill_between(t[day2] - 24, 20.3, 20.3 + 0.15 * mode[day2], color="#C98510", alpha=0.6, step="post")
ax[0].axhline(20.5, ls=":", color="#56616D")
ax[0].axhline(21.5, ls=":", color="#56616D")
ax[0].set_xlabel("hour of the second day")
ax[0].set_ylabel("temperature (C)")
bands = np.linspace(0.1, 3, 60)
ax[1].plot(bands, [48 / sum(cycle_times(b, 5.0)) for b in bands], color="#1E6FA8")
ax[1].set_xlabel("band of the hysteresis (K)")
ax[1].set_ylabel("switchings a day")
plt.tight_layout()
plt.show()

**Observe and reflect.** What does the band of the hysteresis trade against the number of switchings? What would happen to the contactor of the heater with a band of zero?

## 3. The overcurrent relay: Your tests &#91;[7](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#ref-7)&#93;

*Lecture: [Protection: The overcurrent relay](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#protection-relays)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "thermostat" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-03/"
               "NB03_discrete_hybrid_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A short circuit drives the current of a feeder far above its rating, and the overcurrent relay must open the breaker before cables and devices are damaged. Python computes the standard inverse curve, simulates the relay as a hybrid automaton with an internal timer and finds the setting of the upstream relay that keeps the protection selective.

**From the formulas to the code.** The operating time $t = TMS \cdot k / (M^{\alpha} - 1)$ with the time multiplier setting (TMS), $k = 0.14$, $\alpha = 0.02$ and the multiple $M = I/I_s$ is `t_iec(M, tms)`. The relay integrates $1/t(M)$ over time and trips when the integral reaches 1, which is the body of `relay`. The grading margin is the difference between the trip times of the incomer and of the feeder relay.

In [ ]:
def t_iec(M, tms, k=K_SI, alpha=ALPHA_SI):
    M = np.asarray(M, float)
    return np.where(M > 1, tms * k / (np.maximum(M, 1 + 1e-9) ** alpha - 1), np.inf)

def relay(I_amps, I_s, tms, dt=0.001, t_max=60.0):
    """Hybrid automaton NORMAL, PICKUP, TRIP; I_amps gives the current in A at the time t in s."""
    theta, t = 0.0, 0.0
    while t < t_max:
        M = I_amps(t) / I_s
        if M > 1:                          # PICKUP: the timer runs at the rate 1 / t(M)
            theta += dt / float(t_iec(M, tms))
            if theta >= 1:
                return round(t + dt, 3)    # TRIP
        else:
            theta = 0.0                    # NORMAL: an instant reset
        t += dt
    return None

print(f"M = 6, TMS = 0.2: {float(t_iec(6, 0.2)):.3f} s")
for M in [2, 5, 10, 20]:
    print(f"feeder relay, TMS 0.10, M = {M:>2}: {float(t_iec(M, 0.1)):.3f} s")
I_FAULT, IS_FEEDER, IS_INCOMER = 5000.0, 250.0, 1600.0
t_feeder = relay(lambda t: I_FAULT, IS_FEEDER, 0.10)
print(f"fault of {I_FAULT:.0f} A on the feeder of the library: the feeder relay trips after {t_feeder:.3f} s")
margin = 0.3                                          # grading margin of the campus
tms_min = (t_feeder + margin) / float(t_iec(I_FAULT / IS_INCOMER, 1.0))
tms_inc = math.ceil(tms_min * 100) / 100              # relays are set in steps of 0.01
t_incomer = relay(lambda t: I_FAULT, IS_INCOMER, tms_inc)
print(f"incomer: M = {I_FAULT / IS_INCOMER:.3f}, smallest TMS {tms_min:.4f}, set to {tms_inc:.2f}")
print(f"the incomer would trip after {t_incomer:.3f} s, a grading margin of {t_incomer - t_feeder:.3f} s")

**Python note.** `np.where(M > 1, a, b)` chooses element by element between two arrays, so the curve is infinite below the pickup. `math.ceil(x * 100) / 100` rounds up to the next step of 0.01, because a smaller setting would break the margin.

**Explore.** The start of the chiller of the laboratories draws 900 A that decay with a time constant of 0.5 s, on top of 150 A of other load. The relay of the feeder of the laboratories has the same pickup of 250 A. Which time multiplier rides through the start?

In [ ]:
start = lambda t: 150 + 900 * math.exp(-t / 0.5)          # the current of the feeder in A at t seconds
for tms in [0.05, 0.10, 0.15, 0.20]:
    trip = relay(start, IS_FEEDER, tms, t_max=5)
    print(f"TMS {tms:.2f}: " + (f"trips after {trip:.3f} s" if trip else "rides through the start"))

**Exercise C.** Store the trip time of the feeder relay at $M = 10$ with $TMS = 0.10$, in seconds, in `t_c`.

In [ ]:
t_c = None   # your answer

check("Exercise C", t_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_c = float(t_iec(10, 0.1))

check("Exercise C", t_c)

Part B of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lab.html) tests the relays on the bench with a secondary injection test set. Paste the copied rows into `MY_TESTS`. The code fits the time multiplier and the fixed delay of the feeder relay from your measurements.

In [ ]:
# Paste the results of lab Part B between the triple quotes.
MY_TESTS = """
"""
if MY_TESTS.strip():
    tests = pd.read_csv(io.StringIO(MY_TESTS.strip()))
    source = "your own tests on the relay bench"
else:
    tests = pd.DataFrame({"test": [1, 2, 3, 4, 5], "relay": ["feeder"] * 4 + ["incomer"],
                          "multiple": [2, 5, 10, 20, 3.125], "tms": [0.1, 0.1, 0.1, 0.1, 0.09],
                          "trip_s": [1.023, 0.448, 0.317, 0.247, 0.567]})
    source = "a simulated student"
tests["model_s"] = t_iec(tests["multiple"], tests["tms"]).round(3)
print("Data:", source)
print(tests.to_string(index=False))
feeder = tests[(tests["relay"] == "feeder") & np.isclose(tests["tms"], 0.1)]
if feeder["multiple"].nunique() >= 2:
    g = K_SI / (feeder["multiple"] ** ALPHA_SI - 1)        # the curve for TMS = 1
    slope, delay = np.polyfit(g, feeder["trip_s"], 1)
    print(f"fitted TMS {slope:.3f} and fixed delay {1000 * delay:.0f} ms of the feeder relay")
else:
    print("Test the feeder relay with TMS 0.10 at two or more multiples to fit its setting.")

**Python note.** `np.polyfit(x, y, 1)` fits a straight line and returns its slope and intercept. Against the curve for $TMS = 1$, the slope is the time multiplier and the intercept the fixed delay of the output contact of the relay.

In [ ]:
# Figure: the standard inverse curves of the feeder and of the incomer, and the fault of 5 kA
I = np.logspace(np.log10(260), np.log10(20000), 300)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.loglog(I, t_iec(I / IS_FEEDER, 0.10), color="#1E6FA8", label="feeder of the library, 250 A, TMS 0.10")
ax.loglog(I, t_iec(I / IS_INCOMER, tms_inc), color="#B53F2E", label=f"incomer, 1600 A, TMS {tms_inc:.2f}")
ax.axvline(I_FAULT, ls=":", color="#56616D")
ax.set_ylim(0.05, 100)
ax.set_xlabel("current (A)")
ax.set_ylabel("trip time (s)")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** Why must the incomer wait longer than the feeder relay? What would happen in island mode, when the inverter feeds a fault with little more than its rated current?

## 4. The microgrid controller: Your outage &#91;[9](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#ref-9)&#93;

*Lecture: [The microgrid as a hybrid system](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#microgrid-modes)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "tms_inc" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-03/"
               "NB03_discrete_hybrid_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** When the city grid fails, the campus can continue as an island if the battery forms its own grid and the load fits the inverter. Python writes the controller of the microgrid as a state machine with guards on the state of charge and simulates an evening outage minute by minute.

**From the formulas to the code.** The modes of the controller are written in capitals: GRID, DARK, ISLAND, SHED and RESYNC, short for resynchronisation. The guards compare the state of charge with 0.30 and 0.10, and the load $P_{load} - P_{wind}$ with the inverter limit of 250 kW. The state of charge falls by $P\,\Delta t/(\eta E)$ in each minute, as in Week 2. `outage(soc0)` returns the log of the transitions and the state of charge minute by minute. `sync_wait` gives the time until the phase angle enters the window of the synchronisation.

In [ ]:
PEAK_KW = {"B1": 180, "B2": 120, "B3": 150, "B4": 90, "B5": 160}
PRIORITY = ["car_park", "B4", "B1", "B3", "B2", "B5"]    # shed first ... shed last
CRITICAL = ["B2", "B5"]                                  # the library and the laboratories

def wind_power(v, p_rated=250.0, v_in=3.0, v_rated=12.0, v_out=25.0):
    v = np.asarray(v, float)
    p = p_rated * (v ** 3 - v_in ** 3) / (v_rated ** 3 - v_in ** 3)
    p = np.where(v < v_in, 0.0, np.where(v < v_rated, p, p_rated))
    return np.where(v >= v_out, 0.0, p)

def feeders(hour):
    """Load of each feeder in kW: the buildings share the campus load in proportion to their peaks."""
    d = {b: LOAD_KW[hour] * p / sum(PEAK_KW.values()) for b, p in PEAK_KW.items()}
    d["car_park"] = EV_KW[hour]
    return d

def kept_kw(hour):
    """The load that the island keeps after shedding by priority until it fits the inverter."""
    loads, wind = feeders(hour), float(wind_power(WIND_MS[hour]))
    connected = list(PRIORITY)
    while sum(loads[f] for f in connected) - wind > P_INV_KW:
        connected.pop(0)
    return round(sum(loads[f] for f in connected), 1)

def hhmm(t):
    m = int(round(t * 60))
    return f"{m // 60 % 24:02d}:{m % 60:02d}"

def outage(soc0, start_h=21.0, hours=2.0, shed_soc=0.30, empty_soc=0.10, dt_h=1 / 60):
    """Minute-by-minute controller of the island from the loss of the grid until its return."""
    loads, wind = feeders(int(start_h)), float(wind_power(WIND_MS[int(start_h)]))
    connected = list(PRIORITY)
    while sum(loads[f] for f in connected) - wind > P_INV_KW:   # shed until the load fits the inverter
        connected.pop(0)
    soc, mode, curve = soc0, "ISLAND", []
    log = [(start_h, "grid lost", "GRID", "DARK", soc), (start_h, "island formed", "DARK", "ISLAND", soc)]
    for k in range(int(round(hours / dt_h))):
        t = start_h + k * dt_h
        loads, wind = feeders(int(t) % 24), float(wind_power(WIND_MS[int(t) % 24]))
        if mode == "ISLAND" and soc < shed_soc:
            connected = [f for f in connected if f in CRITICAL]
            log.append((t, "shed to critical", "ISLAND", "SHED", soc))
            mode = "SHED"
        if mode in ("ISLAND", "SHED") and soc <= empty_soc:
            log.append((t, "battery empty", mode, "DARK", soc))
            mode, connected = "DARK", []
        curve.append((t, soc, mode))
        if mode != "DARK":
            soc -= min(P_INV_KW, max(0.0, sum(loads[f] for f in connected) - wind)) * dt_h / (ETA * E_KWH)
    t_back = start_h + hours
    if mode in ("ISLAND", "SHED") and soc <= empty_soc:      # the guard holds at the return of the grid as well
        log.append((t_back, "battery empty", mode, "DARK", soc))
        mode = "DARK"
    log.append((t_back, "grid back", mode, "RESYNC" if mode != "DARK" else "GRID", soc))
    return log, curve

def sync_wait(phi0_deg, slip_hz=0.1, window_deg=15.0):
    """Seconds until the phase angle, which turns at 360 * slip degrees a second, enters the window."""
    return ((360 - window_deg) - phi0_deg) % 360 / (360 * slip_hz)

def show(log):
    for t, event, a, b, soc in log:
        print(f"{hhmm(t)}  {event:<17} {a:<7} -> {b:<7} SoC {soc:.3f}")

print(f"kept at 21:00: {kept_kw(21)} kW of {sum(feeders(21).values()):.1f} kW")
log, curve = outage(0.144)
show(log)
print(f"resynchronisation from 120 degrees at a slip of 0.1 Hz: the breaker closes after {sync_wait(120):.2f} s")

**Python note.** `connected.pop(0)` removes the first feeder of the list, the one with the lowest priority, until the load fits the inverter. Each entry of the log is a tuple of the time, the event, the mode before, the mode after and the state of charge. The function returns two values, which `log, curve = ...` unpacks.

**Exercise D.** Store the load in kW that the island keeps at 21:00 after shedding by priority in `kept_d`.

In [ ]:
kept_d = None   # your answer

check("Exercise D", kept_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
kept_d = kept_kw(21)

check("Exercise D", kept_d)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lab.html) puts you at the switchboard of the campus during the same outage. Paste the copied event log into `MY_LOG`. The code replays it through the transition table of the controller and marks every transition that the table does not allow.

In [ ]:
ALLOWED = {("GRID", "grid lost"): "DARK", ("DARK", "island formed"): "ISLAND",
           ("ISLAND", "shed to critical"): "SHED", ("ISLAND", "battery empty"): "DARK",
           ("SHED", "battery empty"): "DARK", ("ISLAND", "overload"): "DARK", ("SHED", "overload"): "DARK",
           ("ISLAND", "grid back"): "RESYNC", ("SHED", "grid back"): "RESYNC", ("DARK", "grid back"): "GRID",
           ("RESYNC", "in phase"): "GRID"}
# Paste the event log of lab Part C between the triple quotes.
MY_LOG = """
"""
if MY_LOG.strip():
    events_log = pd.read_csv(io.StringIO(MY_LOG.strip()))
    source = "your own outages at the switchboard"
else:
    events_log = pd.DataFrame([(1, "21:00", "grid lost", "GRID", "DARK", 0.144),
                               (1, "21:01", "island formed", "DARK", "ISLAND", 0.144),
                               (1, "21:01", "shed to critical", "ISLAND", "SHED", 0.144),
                               (1, "21:10", "battery empty", "SHED", "DARK", 0.098),
                               (1, "23:00", "grid back", "DARK", "GRID", 0.098)],
                              columns=["run", "time", "event", "from", "to", "soc"])
    source = "a simulated student"
def legal(row):
    if row["from"] == row["to"]:
        return "action"                  # breaker operations and blocked closings keep the mode
    return "ok" if ALLOWED.get((row["from"], row["event"])) == row["to"] else "NOT ALLOWED"
events_log["check"] = events_log.apply(legal, axis=1)
print("Data:", source)
print(events_log.to_string(index=False))
minutes = lambda s: int(s[:2]) * 60 + int(s[3:])
for run_id, g in events_log.groupby("run"):
    dark, t_dark = 0, None
    for _, r in g.iterrows():
        if r["to"] == "DARK" and r["from"] != "DARK":
            t_dark = minutes(r["time"])
        elif r["from"] == "DARK" and r["to"] != "DARK" and t_dark is not None:
            dark += minutes(r["time"]) - t_dark
            t_dark = None
    print(f"run {run_id}: {dark} minutes without supply")

**Python note.** `ALLOWED.get((from, event))` looks up the transition that the controller allows. A row that keeps its mode, such as the opening of a feeder breaker, is an action of the operator and not a transition. The minutes without supply add up the intervals from each entry into DARK to the next exit.

**Observe and reflect.** How long does the campus stay without supply in your runs? Which of your actions would an automatic controller take within a second?

## 5. The campus microgrid, stage 3

*Lecture: [The campus microgrid, stage 3: Operating modes and protection](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#the-campus-microgrid-stage-3)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "outage" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-03/"
               "NB03_discrete_hybrid_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 3 collects one result of each discrete or hybrid model. The first is the cycle of the thermostat of the library. The second is the setting of the incomer that keeps the protection selective. The third is the state of charge that the battery needs at 21:00 to ride through an outage of two hours.

In [ ]:
def rides_through(soc0):
    log, _ = outage(soc0)
    return all(event != "battery empty" for _, event, _, _, _ in log)

STAGE3 = {}
t_on, t_off = cycle_times(1.0, 5.0)
STAGE3["thermostat, band 1 K at 5 C: on time (min)"] = round(60 * t_on, 1)
STAGE3["thermostat, band 1 K at 5 C: off time (min)"] = round(60 * t_off, 1)
STAGE3["thermostat: switchings a day"] = round(48 / (t_on + t_off), 1)
STAGE3["feeder relay at 5 kA (s)"] = t_feeder
STAGE3["TMS of the incomer for a margin of 0.3 s"] = tms_inc
STAGE3["grading margin (s)"] = round(t_incomer - t_feeder, 3)
empty = [t for t, event, _, _, _ in outage(0.144)[0] if event == "battery empty"]
STAGE3["outage at 21:00 with SoC 0.144: dark from"] = hhmm(empty[0])
grid = np.round(np.arange(0.10, 0.9005, 0.001), 3)       # every state of charge in steps of 0.001
ok = np.array([rides_through(s0) for s0 in grid])
fails = np.where(~ok)[0]
need = grid[fails[-1] + 1] if ok[-1] else None             # from here on, every higher value rides through
STAGE3["SoC at 21:00 that rides through until 23:00"] = need
for key, value in STAGE3.items():
    print(f"{key:<48} {value}")

**Python note.** `rides_through` is `True` when the log of the outage contains no event `battery empty`. The code tries every state of charge in steps of 0.001 instead of a bisection. The controller acts only at whole minutes, so a slightly higher state of charge can delay the shedding by one minute and cost more energy than it adds. The answer is the smallest value from which every higher value rides through.

In [ ]:
# Figure: the outage of the evening with the state of charge of Week 2 and with the reserve of stage 3
fig, ax = plt.subplots(figsize=(8, 3.4))
for s0, c in [(0.144, "#B53F2E"), (STAGE3["SoC at 21:00 that rides through until 23:00"], "#2E8B57")]:
    log, curve = outage(s0)
    ax.plot([c_[0] for c_ in curve], [c_[1] if c_[2] != "DARK" else np.nan for c_ in curve], color=c,
            label=f"SoC {s0:.3f} at 21:00")
    for t, event, a, b, soc in log[2:-1]:
        ax.plot(t, soc, "o", color=c, ms=5)
ax.axhline(0.30, ls=":", color="#C98510")
ax.axhline(0.10, ls=":", color="#B53F2E")
ax.set_xlabel("hour of the evening")
ax.set_ylabel("state of charge")
ax.legend(fontsize=8)
plt.show()

**Exercise E.** Store the state of charge at 21:00 that rides through the outage, as kept in `STAGE3`, in `soc_e`.

In [ ]:
soc_e = None   # your answer

check("Exercise E", soc_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
soc_e = STAGE3["SoC at 21:00 that rides through until 23:00"]

check("Exercise E", soc_e)

**Observe and reflect.** The evening shaving of Week 2 leaves a state of charge of 0.144 at 21:00. What does the result of stage 3 say about using the battery for saving money and for riding through outages at the same time?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE3" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-03/"
               "NB03_discrete_hybrid_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Many devices of daily life are state machines, such as a traffic light, the door of an elevator or a washing machine. The functions of this week run them after one change of a name.

The machines are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "washing_machine"
# choose: charger, traffic_light, elevator_door, washing_machine, water_pump, vending_machine
APPS = {
    "charger": CHARGER,
    "traffic_light": {("red", "tick"): "red_amber", ("red_amber", "tick"): "green", ("green", "tick"): "amber",
                      ("amber", "tick"): "red", ("red", "fault"): "flashing", ("green", "fault"): "flashing",
                      ("flashing", "repair"): "red"},
    "elevator_door": {("closed", "call"): "opening", ("opening", "end"): "open", ("open", "timer"): "closing",
                      ("closing", "end"): "closed", ("closing", "obstacle"): "opening"},
    "washing_machine": {("idle", "start"): "fill", ("fill", "full"): "wash", ("wash", "timer"): "drain",
                        ("drain", "empty"): "spin", ("spin", "timer"): "idle", ("fill", "door"): "error",
                        ("wash", "door"): "error", ("error", "reset"): "idle"},
    "water_pump": {("off", "low"): "on", ("on", "high"): "off", ("on", "dry"): "alarm", ("alarm", "reset"): "off"},
    "vending_machine": {("idle", "coin"): "paid", ("paid", "coin"): "paid", ("paid", "select"): "vend",
                        ("vend", "done"): "idle", ("paid", "cancel"): "refund", ("refund", "done"): "idle"}}
table = APPS[APP]
s0 = next(iter(table))[0]
events = sorted({e for _, e in table})
rng = np.random.default_rng(RANDOM_STATE)
sequence = list(rng.choice(events, 12))
print(f"{APP}: start in {s0}, events {events}")
print("trace:", " -> ".join(run(table, s0, sequence)))
states = {s for s, _ in table} | set(table.values())
print("reachable:", sorted(reachable(table, s0)), "| unreachable:", sorted(states - reachable(table, s0)))
print("states without a way out:", sorted(s for s in states if not any(src == s for src, _ in table)))

**Python note.** The start state is the first state of the table, and `rng.choice` draws twelve random events. A state without a way out would trap the machine for ever, which a design review should find.

**Observe and reflect.** Which of the machines can get stuck, and which event would you add to rescue it? Which transition would you test first on the real device?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write a class `StateMachine` with a table, a current state and a method `fire(event)` that raises an error for events without a transition, and test it on the charger. |
| Embedded systems | Write the thermostat of Section 2 as it would run on a microcontroller: a loop that reads the temperature once a minute, with the band as an integer in tenths of a degree. |
| Internet of Things | The chargers of the car park report their state over the network once a minute. Simulate ten chargers and count how often the whole car park draws more than 110 kW. |
| Computer networks | The command to open the PCC breaker arrives 0.5 s late. Add this delay to the relay of Section 3 and find the TMS of the incomer that keeps the margin. |
| Control engineering | Replace the hysteresis of the thermostat by a minimum off time of 15 minutes and compare the switchings and the swing of the temperature. |
| Data science | Generate a log of 1000 random events for the charger and estimate the probability of each transition from the log. Which transitions never appear? |
| Cybersecurity | An attacker sends the event `ready` to a charger in state A. Show with the table that nothing happens, and find a sequence of two events that closes the contactor. |
| Artificial intelligence | Learn the transition table of the washing machine from its random traces alone, and compare the learned table with the original. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-03).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>